# Exploration — Audiogrammes Odyo

Ce notebook explore les données brutes, vérifie la qualité et teste les deux pipelines ML.

**Étapes :**
1. Chargement des données JSON
2. Exploration descriptive
3. Construction des features
4. Pipeline non supervisé (Isolation Forest + Autoencoder)
5. Pipeline supervisé (si labels disponibles)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.loader import load_json_file, load_dataset
from src.features import build_feature_matrix, preprocess, STANDARD_FREQS
from src.models.unsupervised import run_unsupervised_pipeline
from src.models.supervised import run_supervised_pipeline
from src.evaluate import (
    plot_anomaly_score_distribution,
    plot_audiogram,
    plot_top_anomalies,
    plot_umap,
    summary_report,
)

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Chargement des données

In [ ]:
# Option A : charger un seul fichier (sample)
records = load_json_file('../sample.json')
df = pd.DataFrame(records)

# Option B : charger tout un dossier
# df = load_dataset('../data/raw/')

print(f"Records chargés : {len(df)}")
df.head()

## 2. Exploration descriptive

In [ ]:
print("=== Colonnes ===")
print(df.dtypes)

print("\n=== Valeurs manquantes ===")
print(df.isnull().sum())

print("\n=== Distribution de 'category' ===")
print(df['category'].value_counts())

print("\n=== Distribution de 'data_type' ===")
print(df['data_type'].value_counts())

In [ ]:
# Visualiser un audiogramme exemple
row = df.iloc[0]
plot_audiogram(row['dots_left'], row['dots_right'], title=f"Exemple — record #{row['record_id']}")
plt.show()

In [ ]:
# Distribution du nombre de fréquences testées
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df['n_freqs_left'].hist(ax=axes[0], bins=15, color='blue', alpha=0.7)
axes[0].set_title('Nb fréquences OG')
df['n_freqs_right'].hist(ax=axes[1], bins=15, color='red', alpha=0.7)
axes[1].set_title('Nb fréquences OD')
plt.tight_layout()
plt.show()

## 3. Feature engineering

In [ ]:
feature_df, feature_cols = build_feature_matrix(df)
X, scaler, imputer = preprocess(feature_df, fit=True)

print(f"Shape matrice de features : {X.shape}")
print(f"Features : {feature_cols}")
feature_df.describe().round(1)

In [ ]:
# Heatmap de corrélation
fig, ax = plt.subplots(figsize=(12, 9))
corr = feature_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, annot_kws={'size': 7})
ax.set_title('Corrélation entre features')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des seuils OG et OD aux fréquences standard
left_cols = [f'L_{f}' for f in STANDARD_FREQS]
right_cols = [f'R_{f}' for f in STANDARD_FREQS]

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i, (lc, rc, freq) in enumerate(zip(left_cols, right_cols, STANDARD_FREQS)):
    feature_df[lc].hist(ax=axes[0, i], bins=20, color='blue', alpha=0.7)
    axes[0, i].set_title(f'OG {freq} Hz')
    feature_df[rc].hist(ax=axes[1, i], bins=20, color='red', alpha=0.7)
    axes[1, i].set_title(f'OD {freq} Hz')
plt.suptitle('Distribution des seuils dB par fréquence', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Pipeline non supervisé

In [ ]:
scores_df, if_model, ae_model = run_unsupervised_pipeline(
    X,
    contamination=0.05,  # ~5% d'anomalies supposées
    ae_epochs=100,
)

summary_report(df, scores_df)

In [ ]:
# Distribution des scores
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_anomaly_score_distribution(scores_df['anomaly_score_if'], title='Isolation Forest', ax=axes[0])
plot_anomaly_score_distribution(scores_df['reconstruction_error'], title='Autoencoder (reconstruction error)', ax=axes[1])
plot_anomaly_score_distribution(scores_df['pca_reconstruction_error'], title='PCA reconstruction error', ax=axes[2])
plt.tight_layout()
plt.show()

In [ ]:
# UMAP coloré par score d'anomalie (Autoencoder)
plot_umap(X, scores_df['reconstruction_error'].values, title='UMAP — Erreur de reconstruction (Autoencoder)', label_name='Reconstruction error')
plt.show()

In [ ]:
# Audiogrammes des top anomalies selon l'Autoencoder
plot_top_anomalies(df, scores_df['reconstruction_error'], n=6, score_col='reconstruction_error')
plt.show()

## 5. Pipeline supervisé (si labels disponibles)

In [ ]:
# Vérifier si 'category' est un label exploitable
print("Distribution de 'category' :")
print(df['category'].value_counts())
print(f"\nValeurs uniques : {df['category'].unique()}")

In [ ]:
# Décommenter et adapter quand les labels sont confirmés

# y = df['category'].values  # ou binariser : (df['category'] != 0).astype(int)

# if len(np.unique(y)) > 1:
#     from src.models.supervised import run_supervised_pipeline
#     from src.evaluate import plot_roc_curve, plot_feature_importance, plot_confusion_matrix
#
#     model, cv_results = run_supervised_pipeline(X, y, model_type='xgboost')
#
#     from sklearn.model_selection import train_test_split
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
#     model.fit(X_train, y_train)
#
#     fig, axes = plt.subplots(1, 2, figsize=(12, 5))
#     plot_confusion_matrix(y_test, model.predict(X_test), ax=axes[0])
#     plot_roc_curve(model, X_test, y_test, ax=axes[1])
#     plt.tight_layout()
#     plt.show()
#
#     plot_feature_importance(model, feature_cols)
#     plt.show()
# else:
#     print("Labels pas encore disponibles — utiliser le pipeline non supervisé.")

print("Section supervisée en attente de confirmation des labels 'category'.")